In [1]:
import json
import time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

In [2]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    ##api_key="9d9240b6592a464399b88dafe710da9c.U00kh_iANaeR5HPen36I0rUd"
    api_key="ollama"
)

In [3]:
INCLUSION = """
    1. Study about Large Language Models for Educational Information Extraction.
"""

EXCLUSION = """
    1. Not related Large Language Models for Educational Information Extraction.
"""

In [4]:
df = pd.read_csv("Scopus.csv")

In [5]:
df.head(5)

,ID,Title,Year,Source title,Abstract,Author Keywords,Document Type
0,1,Enhancing Few-Shot Biomedical Relation Extract...,2025,2025 4th International Conference on Artificia...,Relation extraction plays a key role in biomed...,few-shot learning; large language models; prom...,Conference paper
1,2,AI-assisted interpretation of Markush structur...,2026,Journal of Cheminformatics,Automated interpretation of Markush structures...,Cheminformatics; Information extraction; Large...,Review
2,3,Knowledge graph-enhanced large language models...,2026,Reliability Engineering and System Safety,Crane accidents are characterized by high sudd...,Causal analysis; Crane accidents; Explainable ...,Article
3,4,19th International Conference on Document Anal...,2026,Lecture Notes in Computer Science,The proceedings contain 148 papers. The specia...,NaN,Conference review
4,5,Transformer-Based Models for Natural Language ...,2026,2026 5th International Conference on Electroni...,Transformer architectures have become the domi...,Chinese NLP; large language models; long conte...,Conference paper


In [10]:
import json
import re

def screening(title, abstract):

    prompt = f"""
        Evaluate the following paper for a Systematic Literature Review.

        Inclusion Criteria:
        {INCLUSION}

        Exclusion Criteria:
        {EXCLUSION}

        Title:
        {title}

        Abstract:
        {abstract}

        Return ONLY a valid JSON object with exactly this format:

        {{
            "decision": "Include | Exclude | Uncertain",
            "confidence": 0,
            "reason": "Short reason (max 30 words)"
        }}
        """

    try:

        response = client.chat.completions.create(
            model="gemma3:4b",
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert systematic literature reviewer. "
                        "Never explain your reasoning. "
                        "Never use markdown. "
                        "Return ONLY valid JSON."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        content = response.choices[0].message.content.strip()

        # Hapus markdown
        content = content.replace("```json", "")
        content = content.replace("```", "")

        # Hapus <think> jika ada
        content = re.sub(
            r"<think>.*?</think>",
            "",
            content,
            flags=re.DOTALL
        ).strip()

        # Ambil JSON pertama
        match = re.search(r"\{.*\}", content, re.DOTALL)

        if match:
            content = match.group(0)

        result = json.loads(content)

        # Validasi field
        result.setdefault("decision", "ERROR")
        result.setdefault("confidence", 0)
        result.setdefault("reason", "")

        return result

    except Exception as e:

        return {
            "decision": "ERROR",
            "confidence": 0,
            "reason": str(e),
            "raw_output": content if "content" in locals() else ""
        }

In [11]:
paper = df.iloc[0]

result = screening(
    paper["Title"],
    paper["Abstract"]
)

result

{'decision': 'Include',
 'confidence': 0.95,
 'reason': 'The study focuses on Large Language Models for Educational Information Extraction (biomedical relation extraction).'}

In [12]:
results = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    result = screening(
        row["Title"],
        row["Abstract"]
    )

results.append(result)

  0%|          | 0/844 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
result_df = pd.DataFrame(results)

output = pd.concat(
    [df, result_df],
    axis=1
)

,ID,Title,Year,Source title,Abstract,Author Keywords,Document Type
0,1,Enhancing Few-Shot Biomedical Relation Extract...,2025,2025 4th International Conference on Artificia...,Relation extraction plays a key role in biomed...,few-shot learning; large language models; prom...,Conference paper
1,2,AI-assisted interpretation of Markush structur...,2026,Journal of Cheminformatics,Automated interpretation of Markush structures...,Cheminformatics; Information extraction; Large...,Review
2,3,Knowledge graph-enhanced large language models...,2026,Reliability Engineering and System Safety,Crane accidents are characterized by high sudd...,Causal analysis; Crane accidents; Explainable ...,Article
3,4,19th International Conference on Document Anal...,2026,Lecture Notes in Computer Science,The proceedings contain 148 papers. The specia...,NaN,Conference review
4,5,Transformer-Based Models for Natural Language ...,2026,2026 5th International Conference on Electroni...,Transformer architectures have become the domi...,Chinese NLP; large language models; long conte...,Conference paper


In [17]:
output

,ID,Title,Year,Source title,Abstract,Author Keywords,Document Type
0,1,Enhancing Few-Shot Biomedical Relation Extract...,2025,2025 4th International Conference on Artificia...,Relation extraction plays a key role in biomed...,few-shot learning; large language models; prom...,Conference paper
1,2,AI-assisted interpretation of Markush structur...,2026,Journal of Cheminformatics,Automated interpretation of Markush structures...,Cheminformatics; Information extraction; Large...,Review
2,3,Knowledge graph-enhanced large language models...,2026,Reliability Engineering and System Safety,Crane accidents are characterized by high sudd...,Causal analysis; Crane accidents; Explainable ...,Article
3,4,19th International Conference on Document Anal...,2026,Lecture Notes in Computer Science,The proceedings contain 148 papers. The specia...,NaN,Conference review
4,5,Transformer-Based Models for Natural Language ...,2026,2026 5th International Conference on Electroni...,Transformer architectures have become the domi...,Chinese NLP; large language models; long conte...,Conference paper
...,...,...,...,...,...,...,...
839,840,Potential of Large Language Models in Health C...,2024,Journal of Medical Internet Research,Background: A large language model (LLM) is a ...,artificial intelligence; attitude; attitudes; ...,Article
840,841,One-shot Biomedical Named Entity Recognition v...,2024,ACM-BCB 2024 - 15th ACM Conference on Bioinfor...,Large Language Models (LLMs) have demonstrated...,Biomedical Named Entity Recognition; Large Lan...,Conference paper
841,842,Impact of Sample Selection on In-Context Learn...,2023,Findings of the Association for Computational ...,Prompt-based usage of Large Language Models (L...,NaN,Conference paper
842,843,GLM-4-Based Method for Automatic Construction ...,2025,IEEE Access,Content graphs are essential for representing ...,Course content graph; digital teaching platfor...,Article


In [14]:
output.to_csv(
    "screened_results2.csv",
    index=False
)

print("Finished")

Finished
